# 🏆 Master‑Level Chess Policy + Value Network (Dual‑Head)
**Elite game filtering & dual‑task learning**  
Trains a policy head (move selection) and a value head (position evaluation) on high‑quality PGNs.

---
### Highlights
- Games filtered by Elo ≥ 2500 **or** from named elite players  
- 19‑plane board tensor + 73‑plane policy encoding  
- Shared residual trunk, separate policy & value heads  
- Combined loss: cross‑entropy (policy) + MSE (value)  
- ~4 million parameters, regularised with dropout & weight decay

In [2]:
# ========================
# IMPORTS
# ========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import chess.pgn
import chess
import numpy as np
import random
import os

print('Libraries imported successfully.')

Libraries imported successfully.


## ⚙️ Configuration
Adjust file paths and training parameters here.

In [3]:
# ========================
# CONFIGURATION
# ========================
PGN_FILE = "D://mohammad//Programming//Pytthon_4_AI//NTI_Project - Copy//combined.pgn"          # Your compiled elite/master PGN file
MIN_ELO_THRESHOLD = 2500               # Automatically accept any games above this Elo
ELITE_PLAYERS = {"Tal, Mikhail", "Firouzja, Alireza", "Kasparov, Garry", "Carlsen, Magnus"}

BATCH_SIZE = 256                       # GPU batch size (scaled down for CPU)
EPOCHS = 40
LEARNING_RATE = 1e-3
LABEL_SMOOTHING = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


c:\Users\Omnya\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\cuda\__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 9010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


## 🧠 Board Representation & Move Encoding
Identical to the single‑head version: 19 input planes, 4672 policy outputs.

In [4]:
# 19-plane board tensor
def board_to_tensor(board: chess.Board) -> torch.Tensor:
    piece_map = {chess.PAWN:0, chess.KNIGHT:1, chess.BISHOP:2, chess.ROOK:3, chess.QUEEN:4, chess.KING:5}
    tensor = torch.zeros(19, 8, 8, dtype=torch.float32)

    for sq, p in board.piece_map().items():
        row, col = divmod(sq, 8)
        ch = piece_map[p.piece_type] if p.color == chess.WHITE else 6+piece_map[p.piece_type]
        tensor[ch, row, col] = 1.0

    if board.turn == chess.WHITE:
        tensor[12,:,:] = 1.0

    if board.ep_square is not None:
        r, c = divmod(board.ep_square, 8)
        tensor[13, r, c] = 1.0

    cr = board.castling_rights
    tensor[14,:,:] = 1.0 if cr & chess.BB_H1 else 0.0
    tensor[15,:,:] = 1.0 if cr & chess.BB_A1 else 0.0
    tensor[16,:,:] = 1.0 if cr & chess.BB_H8 else 0.0
    tensor[17,:,:] = 1.0 if cr & chess.BB_A8 else 0.0
    tensor[18,:,:] = board.halfmove_clock / 100.0
    return tensor


# Move → flat policy index (0‑4671)
def move_to_policy_index(move: chess.Move):
    fr, fc = divmod(move.from_square, 8)
    tr, tc = divmod(move.to_square, 8)
    dr, dc = tr - fr, tc - fc

    knight_offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
    if (abs(dr), abs(dc)) in [(1,2), (2,1)]:
        try:
            return 56 + knight_offsets.index((dr, dc)), tr, tc
        except ValueError:
            return None

    dirs = [(-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1), (-1,-1)]
    for dir_idx, (ddr, ddc) in enumerate(dirs):
        if (ddr == 0 and ddc == 0) or (dr == 0 and dc == 0):
            continue
        if ddr != 0 and ddc != 0:
            if abs(dr) != abs(dc): continue
            if dr//abs(dr) != ddr or dc//abs(dc) != ddc: continue
            dist = abs(dr)
        elif ddr == 0:
            if dr != 0: continue
            if dc//abs(dc) != ddc: continue
            dist = abs(dc)
        else:
            if dc != 0: continue
            if dr//abs(dr) != ddr: continue
            dist = abs(dr)
        if 1 <= dist <= 7:
            return dir_idx * 7 + (dist - 1), tr, tc

    if move.promotion and move.promotion != chess.QUEEN:
        if tr == 7:        # White moves UP to promote (dr = +1)
            forward_dr, left_dc, right_dc = 1, -1, 1
        elif tr == 0:      # Black moves DOWN to promote (dr = -1)
            forward_dr, left_dc, right_dc = -1, 1, -1
        else:
            return None
        if dr == forward_dr and dc == 0:
            dir_idx = 1
        elif dr == forward_dr and dc == left_dc:
            dir_idx = 0
        elif dr == forward_dr and dc == right_dc:
            dir_idx = 2
        else:
            return None
        piece_offset = 0 if move.promotion == chess.KNIGHT else (3 if move.promotion == chess.BISHOP else 6)
        return 64 + piece_offset + dir_idx, tr, tc
    return None

## 📂 Dual‑Head Dataset Pipeline
Extracts policy targets **and** game‑outcome‑based value targets for every position.

In [5]:
class DualHeadDataset(Dataset):
    """Returns (board_tensor, policy_target, value_target) for each position."""
    def __init__(self, positions, policy_targets, value_targets):
        self.positions = positions
        self.policy_targets = policy_targets
        self.value_targets = value_targets

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, idx):
        return (
            self.positions[idx],
            torch.tensor(self.policy_targets[idx], dtype=torch.long),
            torch.tensor(self.value_targets[idx], dtype=torch.float32)
        )


def parse_master_games(file_path):
    games_list = []
    
    # Helper to clean weird Elo formats like "2690:2222", "2400?", or missing tags
    def safe_parse_elo(elo_str):
        if not elo_str or elo_str in ["?", "-", "0"]:
            return 0
        # Split at a colon if it exists, and filter out any non-numeric characters
        cleaned = "".join(c for c in elo_str.split(':')[0] if c.isdigit())
        return int(cleaned) if cleaned else 0

    with open(file_path, encoding='utf-8', errors='ignore') as f:
        while True:
            game = chess.pgn.read_game(f)
            if game is None: break
            
            white = game.headers.get("White", "")
            black = game.headers.get("Black", "")
            
            # FIX: Safely parse Elo ratings without throwing a ValueError
            white_elo = safe_parse_elo(game.headers.get("WhiteElo", "0"))
            black_elo = safe_parse_elo(game.headers.get("BlackElo", "0"))
            result_str = game.headers.get("Result", "*")
            
            # Filter Rule: Keep game if it matches target names OR crosses the Elite Elo ceiling
            is_elite_match = (white_elo >= MIN_ELO_THRESHOLD and black_elo >= MIN_ELO_THRESHOLD) or \
                             any(p in white for p in ELITE_PLAYERS) or any(p in black for p in ELITE_PLAYERS)
            
            if not is_elite_match or result_str == "*":
                continue
                
            # Map Game Outcomes: Perspective of White (+1 Win, -1 Loss, 0 Draw)
            if result_str == "1-0": game_outcome = 1.0
            elif result_str == "0-1": game_outcome = -1.0
            else: game_outcome = 0.0
            
            games_list.append((game, game_outcome))
    return games_list


def extract_dual_targets(games_subset):
    """
    For every game, walk through moves and record:
      - board tensor
      - policy index of the played move
      - value = game_outcome (if White) or -game_outcome (if Black)
    """
    positions, policy_targets, value_targets = [], [], []
    for game, game_outcome in games_subset:
        board = game.board()
        for move in game.mainline_moves():
            tensor = board_to_tensor(board)
            idx = move_to_policy_index(move)
            if idx is not None:
                plane, r, c = idx
                flat = plane * 64 + r * 8 + c
                positions.append(tensor)
                policy_targets.append(flat)

                # Value relative to side to move
                current_value = game_outcome if board.turn == chess.WHITE else -game_outcome
                value_targets.append(current_value)
            board.push(move)
    return positions, policy_targets, value_targets

## 🏗️ Dual‑Head Residual CNN
Shared trunk → separate policy head (move probabilities) and value head (position score in [-1, +1]).

In [6]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x + residual)
        return x


class DualHeadChessNet(nn.Module):
    def __init__(self, input_channels=19, num_blocks=6):
        super().__init__()
        # Shared body
        self.conv_input = nn.Sequential(
            nn.Conv2d(input_channels, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(128) for _ in range(num_blocks)])

        # Policy head: 73 planes → flattened logits
        self.policy_head = nn.Sequential(
            nn.Conv2d(128, 73, kernel_size=1, bias=False),
            nn.BatchNorm2d(73),
            nn.ReLU(),
            nn.Flatten(),
            nn.Dropout(0.3)
        )

        # Value head: output a single scalar in [-1, 1]
# Value head: optimized to preserve gradient directional signals
        self.value_head = nn.Sequential(
            nn.Conv2d(128, 1, kernel_size=1, bias=False),
            nn.BatchNorm2d(1),
            nn.LeakyReLU(0.1),  # Retains essential negative advantage metrics
            nn.Flatten(),
            nn.Linear(8 * 8, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.conv_input(x)
        x = self.res_blocks(x)
        policy_logits = self.policy_head(x)
        value_score = self.value_head(x).squeeze(-1)
        return policy_logits, value_score


# Quick parameter count
dummy = DualHeadChessNet()
print(f"Total parameters: {sum(p.numel() for p in dummy.parameters()):,}")

Total parameters: 1,808,533


## 🎯 Dual‑Loss Training Loop
Combines cross‑entropy (policy) and MSE (value) losses with automatic mixed precision.

In [7]:
def train_dual_engine(model, train_loader, val_loader, epochs, lr, device):
    model = model.to(device)
    policy_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    value_criterion = nn.MSELoss()

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    use_amp = (device.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda') if use_amp else None
    
    # Scale coefficient to prevent Policy from drowning out Value learning
    VALUE_WEIGHT = 0.5 

    for epoch in range(epochs):
        model.train()
        total_loss, pol_correct, total = 0, 0, 0

        for x, y_policy, y_value in train_loader:
            x = x.to(device, non_blocking=True)
            y_policy = y_policy.to(device, non_blocking=True)
            y_value = y_value.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast('cuda'):
                    policy_out, value_out = model(x)
                    loss_policy = policy_criterion(policy_out, y_policy)
                    loss_value = value_criterion(value_out, y_value)
                    loss = loss_policy + (VALUE_WEIGHT * loss_value)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                policy_out, value_out = model(x)
                loss_policy = policy_criterion(policy_out, y_policy)
                loss_value = value_criterion(value_out, y_value)
                loss = loss_policy + (VALUE_WEIGHT * loss_value)
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            _, pred = torch.max(policy_out, 1)
            pol_correct += (pred == y_policy).sum().item()
            total += y_policy.size(0)

        train_acc = pol_correct / total
        scheduler.step()

        # Comprehensive Validation
        model.eval()
        val_pol_correct, val_total = 0, 0
        total_val_p_loss, total_val_v_loss = 0, 0
        
        with torch.no_grad():
            for x, y_policy, y_value in val_loader:
                x = x.to(device, non_blocking=True)
                y_policy = y_policy.to(device, non_blocking=True)
                y_value = y_value.to(device, non_blocking=True)

                policy_out, value_out = model(x)
                loss_p = policy_criterion(policy_out, y_policy)
                loss_v = value_criterion(value_out, y_value)
                
                total_val_p_loss += loss_p.item() * x.size(0)
                total_val_v_loss += loss_v.item() * x.size(0)

                _, pred = torch.max(policy_out, 1)
                val_pol_correct += (pred == y_policy).sum().item()
                val_total += y_policy.size(0)

        print(f"Epoch {epoch+1:02d}/{epochs} | Total Train Loss: {total_loss/total:.3f} | "
              f"Train Move Acc: {train_acc:.4f} | Val Move Acc: {val_pol_correct/val_total:.4f} | "
              f"Val Move Loss: {total_val_p_loss/val_total:.3f} | Val Position MSE: {total_val_v_loss/val_total:.3f}")

## 🚀 Main Pipeline
1. Parse & filter master games  
2. Extract dual‑head targets  
3. Create data loaders (auto‑detects CPU / GPU)  
4. Train and save the model

In [ ]:
# ========================
# MAIN EXECUTION
# ========================
if not os.path.exists(PGN_FILE):
    print(f"File missing: Please create '{PGN_FILE}' and load your master files inside.")
else:
    print("Parsing master database structure...")
    elite_games = parse_master_games(PGN_FILE)
    print(f"Total verified master matches filtered: {len(elite_games)}")

    if len(elite_games) == 0:
        print("No games passed the filter. Check your PGN or threshold.")
    else:
        random.seed(42)
        random.shuffle(elite_games)
        split_idx = int(0.8 * len(elite_games))
        train_games = elite_games[:split_idx]
        val_games = elite_games[split_idx:]

        print("Converting training games to tensor vectors...")
        t_pos, t_pol, t_val = extract_dual_targets(train_games)
        print(f"Training states: {len(t_pos)}")

        print("Converting validation games to tensor vectors...")
        v_pos, v_pol, v_val = extract_dual_targets(val_games)
        print(f"Validation states: {len(v_pos)}")

        train_set = DualHeadDataset(t_pos, t_pol, t_val)
        val_set = DualHeadDataset(v_pos, v_pol, v_val)

        # Accelerator check
        cuda_is_working = torch.cuda.is_available() and (torch.cuda.get_device_capability(0)[0] >= 3)
        DEVICE = torch.device("cuda" if cuda_is_working else "cpu")
        CURRENT_BATCH_SIZE = BATCH_SIZE if cuda_is_working else 64

        train_loader = DataLoader(
            train_set, batch_size=CURRENT_BATCH_SIZE, shuffle=True,
            num_workers=0, pin_memory=cuda_is_working
        )
        val_loader = DataLoader(
            val_set, batch_size=CURRENT_BATCH_SIZE, shuffle=False,
            num_workers=0, pin_memory=cuda_is_working
        )

        model = DualHeadChessNet(input_channels=19, num_blocks=6)
        print(f"Dual-Head Engine Configuration Parameters: {sum(p.numel() for p in model.parameters()):,}")

        # Train
        train_dual_engine(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, device=DEVICE)

        # Save
        torch.save(model.state_dict(), "master_resnet.pth")
        print("✨ Process complete! 'master_resnet.pth' saved successfully.")

---
✅ **Done.** The trained dual‑head model can be used for move suggestion **and** position evaluation. Modify the elite player list or Elo threshold as needed.